# 02 — BDD100K Preparation for YOLO Training

**Goal:** Convert BDD100K detection annotations to YOLO format and build the dataset structure.

This notebook:
1. Reads BDD100K detection JSON annotations
2. Converts to YOLO format (`class_id x_center y_center width height`)
3. Creates `images/train`, `images/val`, `labels/train`, `labels/val` locally on Colab SSD
4. Generates the dataset YAML for training
5. Visualises sample annotations
6. Prints class mapping and distribution

## 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python matplotlib Pillow pyyaml tqdm

## 2 — Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
BDD_RAW_DIR = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_raw")
YOLO_DATASET_DIR = "/content/bdd100k_yolo"
DEBUG_LIMIT = None  # Change to 100 for fast debugging

if DEBUG_LIMIT:
    print(f"⚡ DEBUG MODE: Processing only {DEBUG_LIMIT} images")
print(f"Raw BDD dir:     {BDD_RAW_DIR}")
print(f"YOLO dataset:    {YOLO_DATASET_DIR}  (Colab Local SSD)")

## 3 — Clone/Upload Source Utilities

In [ ]:
!git clone https://github.com/Siyun-Chen/EcoCAR-Perception-Pipeline-YOLO26-BDD100K.git /content/ecocar
PROJECT_SRC = "/content/ecocar/src"
if os.path.isdir(PROJECT_SRC):
    sys.path.insert(0, os.path.dirname(PROJECT_SRC))
    print(f"✅ Added {PROJECT_SRC} to path")
else:
    print(f"⚠️ Source dir not found, using inline definitions below.")

In [ ]:
try:
    from src.dataset_utils import (
        BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS,
        convert_bdd100k_to_yolo, create_dataset_yaml,
        verify_dataset_structure, link_or_copy_images,
        print_class_distribution, get_bdd_class_mapping,
        find_expected_images,
    )
    print("✅ Imported from src.dataset_utils")
except ImportError:
    print("⚠️ Could not import src.dataset_utils, using inline definitions")

    import json, shutil
    from pathlib import Path
    import yaml
    from tqdm import tqdm

    BDD_CLASSES = [
        "person", "rider", "car", "truck", "bus",
        "train", "motorcycle", "bicycle", "traffic light", "traffic sign",
    ]
    CLASS_TO_ID = {name: idx for idx, name in enumerate(BDD_CLASSES)}
    ID_TO_CLASS = {idx: name for idx, name in enumerate(BDD_CLASSES)}

    def get_bdd_class_mapping():
        return BDD_CLASSES, CLASS_TO_ID, ID_TO_CLASS

    def convert_bdd100k_to_yolo(json_path, output_label_dir, img_width=1280, img_height=720, debug_limit=None):
        os.makedirs(output_label_dir, exist_ok=True)
        with open(json_path, 'r') as f:
            data = json.load(f)
        if debug_limit:
            data = data[:debug_limit]
        class_counts = {c: 0 for c in BDD_CLASSES}
        for frame in tqdm(data, desc="Converting"):
            stem = Path(frame['name']).stem
            label_path = os.path.join(output_label_dir, stem + '.txt')
            lines = []
            for label in (frame.get('labels') or []):
                cat = label.get('category', '')
                if cat not in CLASS_TO_ID: continue
                box = label.get('box2d')
                if not box: continue
                x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
                xc = max(0, min(1, ((x1+x2)/2) / img_width))
                yc = max(0, min(1, ((y1+y2)/2) / img_height))
                w = max(0, min(1, (x2-x1) / img_width))
                h = max(0, min(1, (y2-y1) / img_height))
                lines.append(f"{CLASS_TO_ID[cat]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
                class_counts[cat] += 1
            with open(label_path, 'w') as f:
                f.write('\n'.join(lines))
        return class_counts

    def create_dataset_yaml(dataset_root, output_path, train_images='images/train', val_images='images/val'):
        config = {'path': dataset_root, 'train': train_images, 'val': val_images,
                  'nc': len(BDD_CLASSES), 'names': {i: n for i, n in enumerate(BDD_CLASSES)}}
        os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
        with open(output_path, 'w') as f:
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)
        print(f"✅ YAML written to: {output_path}")
        return os.path.abspath(output_path)

    def verify_dataset_structure(dataset_root):
        all_ok = True
        for sub in ['images/train','images/val','labels/train','labels/val']:
            full = os.path.join(dataset_root, sub)
            if not os.path.isdir(full):
                print(f"❌ Missing: {full}"); all_ok = False
            else:
                c = len(os.listdir(full))
                print(f"✅ {sub}: {c} files")
                if c == 0: all_ok = False
        return all_ok

    def link_or_copy_images(src_dir, dst_dir, use_symlinks=True, debug_limit=None, image_list=None):
        os.makedirs(dst_dir, exist_ok=True)
        if image_list is not None:
            files = image_list
        else:
            try:
                files = sorted([f for f in os.listdir(src_dir)
                                if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            except OSError:
                print(f"⚠ OSError listing {src_dir}.")
                return 0
        if debug_limit:
            files = files[:debug_limit]
        count = 0
        missing = 0
        for f in tqdm(files, desc=f"→ {os.path.basename(dst_dir)}"):
            src = os.path.join(src_dir, f)
            dst = os.path.join(dst_dir, f)
            if not os.path.exists(src):
                missing += 1; continue
            if os.path.exists(dst):
                count += 1; continue
            ok = False
            try:
                if use_symlinks:
                    os.symlink(src, dst)
                else:
                    shutil.copy2(src, dst)
                ok = True
            except OSError:
                try:
                    shutil.copy2(src, dst)
                    ok = True
                except FileNotFoundError:
                    ok = False
            if ok and os.path.exists(dst):
                count += 1
            else:
                missing += 1
        print(f"Linked/copied: {count}, Missing/skipped: {missing}")
        return count

    def find_expected_images(label_dir, image_src_dirs):
        """Find images matching labels using probe-based FUSE-safe search.

        Avoids os.listdir() on Drive dirs (crashes with ~40K files) and
        avoids blind per-file os.path.exists() x 20K (FUSE timeout).
        Instead: probe 10 files to find a working dir, then full match.
        """
        if isinstance(image_src_dirs, str):
            image_src_dirs = [image_src_dirs]

        # Deprioritize symlinks (FUSE errors on Google Drive)
        real_dirs = [d for d in image_src_dirs if os.path.isdir(d) and not os.path.islink(d)]
        link_dirs = [d for d in image_src_dirs if os.path.isdir(d) and os.path.islink(d)]
        ordered_dirs = real_dirs + link_dirs

        if not ordered_dirs:
            print(f"⚠ No valid dirs among: {image_src_dirs}")
            return [], ''

        # Get label stems from LOCAL dir (fast Colab SSD)
        label_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.txt')])
        label_stems = [os.path.splitext(f)[0] for f in label_files]
        print(f"  📝 {len(label_stems)} label files to match")
        if not label_stems:
            return [], ''

        # Probe: test 10 files per dir to find one that works
        PROBE_SIZE = 10
        probe_stems = label_stems[:PROBE_SIZE]
        chosen_dir = ''
        for d in ordered_dirs:
            hits = 0
            for stem in probe_stems:
                for ext in ['.jpg', '.jpeg', '.png']:
                    if os.path.exists(os.path.join(d, stem + ext)):
                        hits += 1
                        break
            if hits > 0:
                print(f"  ✅ Probe: {hits}/{PROBE_SIZE} found in {d}")
                chosen_dir = d
                break
            else:
                print(f"  ❌ Probe: 0/{PROBE_SIZE} found in {d}")

        if not chosen_dir:
            print('⚠ No directory responded to probe.')
            return [], ''

        # Full match against the working directory
        print(f"  🔍 Full matching against: {chosen_dir}")
        expected = []
        for stem in tqdm(label_stems, desc='Matching'):
            for ext in ['.jpg', '.jpeg', '.png']:
                candidate = stem + ext
                if os.path.exists(os.path.join(chosen_dir, candidate)):
                    expected.append(candidate)
                    break
        return expected, chosen_dir

    def print_class_distribution(class_counts):
        total = sum(class_counts.values())
        print(f"\n{'Class':<20} {'Count':>8} {'Pct':>7}")
        print('─'*37)
        for name in BDD_CLASSES:
            c = class_counts.get(name, 0)
            pct = (c/total*100) if total else 0
            print(f"{name:<20} {c:>8,} {pct:>6.1f}%")
        print('─'*37)
        print(f"{'TOTAL':<20} {total:>8,}")

## 4 — Locate BDD100K Files

In [ ]:
import glob, yaml

paths_cfg_path = os.path.join(ECOCAR_ROOT, 'paths_config.yaml')
TRAIN_LABEL = None
VAL_LABEL = None
BDD_IMAGES_BASE = None

if os.path.isfile(paths_cfg_path):
    with open(paths_cfg_path, 'r') as f:
        pcfg = yaml.safe_load(f)
    for key in pcfg:
        val = pcfg[key]
        if isinstance(val, str) and os.path.isfile(val):
            if 'train' in key and val.endswith('.json'):
                TRAIN_LABEL = val
            elif 'val' in key and val.endswith('.json'):
                VAL_LABEL = val
    if TRAIN_LABEL:
        print(f'✅ Loaded label paths from {paths_cfg_path}')

# Fallback: auto-detect labels
if not TRAIN_LABEL or not VAL_LABEL:
    for p in [
        os.path.join(BDD_RAW_DIR, 'labels', 'bdd100k_labels_images_train.json'),
        os.path.join(BDD_RAW_DIR, 'labels', 'bdd100k_labels_images_val.json'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'bdd100k_labels_images_train.json'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'labels', 'bdd100k_labels_images_val.json'),
    ]:
        if os.path.isfile(p):
            if 'train' in os.path.basename(p) and not TRAIN_LABEL:
                TRAIN_LABEL = p
            elif 'val' in os.path.basename(p) and not VAL_LABEL:
                VAL_LABEL = p

# Auto-detect image base: prefer REAL dirs over symlinks
# 100k/ is real; bdd100k/images/100k/ may be symlinks that break FUSE
if not BDD_IMAGES_BASE:
    for p in [
        os.path.join(BDD_RAW_DIR, '100k'),
        os.path.join(BDD_RAW_DIR, 'images', '100k'),
        os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k'),
    ]:
        if os.path.isdir(p) and (
            os.path.isdir(os.path.join(p, 'train')) or
            os.path.isdir(os.path.join(p, 'val'))
        ):
            BDD_IMAGES_BASE = p
            break

print(f"Train labels: {TRAIN_LABEL}")
print(f"Val labels:   {VAL_LABEL}")
print(f"Images base:  {BDD_IMAGES_BASE}")
if all([TRAIN_LABEL, VAL_LABEL, BDD_IMAGES_BASE]):
    print('\n✅ All paths detected!')
else:
    print('\n❌ Could not auto-detect all paths. Set manually above.')

## 5 — Convert Annotations to YOLO Format

In [ ]:
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATASET_DIR, 'labels', split), exist_ok=True)
print("📁 YOLO local dataset directories created")

In [ ]:
print("="*50 + "\n Converting TRAIN labels\n" + "="*50)
train_counts = convert_bdd100k_to_yolo(
    json_path=TRAIN_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'train'),
    debug_limit=DEBUG_LIMIT,
)
print("\nTrain class distribution:")
print_class_distribution(train_counts)

In [ ]:
print("="*50 + "\n Converting VAL labels\n" + "="*50)
val_counts = convert_bdd100k_to_yolo(
    json_path=VAL_LABEL,
    output_label_dir=os.path.join(YOLO_DATASET_DIR, 'labels', 'val'),
    debug_limit=DEBUG_LIMIT,
)
print("\nVal class distribution:")
print_class_distribution(val_counts)

## 6 — Link/Copy Images (FUSE-Safe)

Uses a probe-based approach to avoid Google Drive FUSE crashes:
- No `os.listdir()` on Drive directories (crashes with ~40K files)
- Probes 10 files first to find a working directory
- Then does full matching via direct `os.path.exists()` per label

In [ ]:
# ── Build candidate image source directories ──────────────────────
# FUSE-safe diagnostics: probe-based, NO os.listdir() on Drive
print("=" * 60)
print(" IMAGE SOURCE DIAGNOSTICS (probe-based, FUSE-safe)")
print("=" * 60)

CANDIDATE_IMG_DIRS = []
all_candidates = []

if BDD_IMAGES_BASE:
    all_candidates.extend([
        os.path.join(BDD_IMAGES_BASE, 'train'),
        os.path.join(BDD_IMAGES_BASE, 'val'),
    ])
extra = [
    os.path.join(BDD_RAW_DIR, '100k', 'train'),
    os.path.join(BDD_RAW_DIR, '100k', 'val'),
    os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k', 'train'),
    os.path.join(BDD_RAW_DIR, 'bdd100k', 'images', '100k', 'val'),
]
for c in extra:
    if c not in all_candidates:
        all_candidates.append(c)

# Probe each dir with a sample file to test responsiveness
# Use a known BDD100K filename pattern for probing
for d in all_candidates:
    if not os.path.isdir(d):
        print(f"  ❌ MISSING: {d}")
        continue
    is_link = os.path.islink(d)
    tag = "SYMLINK" if is_link else "REAL DIR"
    # Quick probe: can we access a file in this dir?
    try:
        # Try os.listdir with a timeout-safe approach
        # If it hangs, we'll catch it in actual matching
        probe_ok = os.access(d, os.R_OK)
        if probe_ok:
            print(f"  {'🔗' if is_link else '✅'} {tag}: {d}  (accessible)")
            CANDIDATE_IMG_DIRS.append(d)
        else:
            print(f"  ⚠️ {tag}: {d}  (not readable)")
    except OSError as e:
        print(f"  ⚠️ {tag}: {d}  (error: {e})")

# Sort: real dirs first, symlinks last
real = [d for d in CANDIDATE_IMG_DIRS if not os.path.islink(d)]
links = [d for d in CANDIDATE_IMG_DIRS if os.path.islink(d)]
CANDIDATE_IMG_DIRS = real + links
print(f"\n📋 {len(CANDIDATE_IMG_DIRS)} candidate dirs (real dirs prioritized)")

In [ ]:
# ── Link TRAIN images ──────────────────────────────────────────────
train_label_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'train')
train_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'train')

print(f"Searching for train images in {len(CANDIDATE_IMG_DIRS)} candidate dirs...")
expected_train_imgs, train_src_dir = find_expected_images(
    train_label_dir, CANDIDATE_IMG_DIRS,
)
print(f"\n✅ Found {len(expected_train_imgs)} matching train images")
if train_src_dir:
    print(f"   Source: {train_src_dir}")

n_train = 0
if expected_train_imgs and train_src_dir:
    n_train = link_or_copy_images(
        train_src_dir, train_img_dst,
        use_symlinks=True, debug_limit=DEBUG_LIMIT,
        image_list=expected_train_imgs,
    )
print(f"✅ {n_train} train images linked")

In [ ]:
# ── Link VAL images ────────────────────────────────────────────────
# Val images may live in the SAME folder as train (nb01's 85/15 split)
val_label_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'val')
val_img_dst = os.path.join(YOLO_DATASET_DIR, 'images', 'val')

print(f"Searching for val images in {len(CANDIDATE_IMG_DIRS)} candidate dirs...")
expected_val_imgs, val_src_dir = find_expected_images(
    val_label_dir, CANDIDATE_IMG_DIRS,
)
print(f"\n✅ Found {len(expected_val_imgs)} matching val images")
if val_src_dir:
    print(f"   Source: {val_src_dir}")

n_val = 0
if expected_val_imgs and val_src_dir:
    n_val = link_or_copy_images(
        val_src_dir, val_img_dst,
        use_symlinks=True, debug_limit=DEBUG_LIMIT,
        image_list=expected_val_imgs,
    )
print(f"✅ {n_val} val images linked")

## 7 — Generate Dataset YAML

In [ ]:
import shutil
yaml_path = os.path.join(YOLO_DATASET_DIR, 'bdd100k_yolo.yaml')
yaml_backup = os.path.join(ECOCAR_ROOT, 'datasets', 'bdd100k_yolo.yaml')
yaml_file = create_dataset_yaml(dataset_root=YOLO_DATASET_DIR, output_path=yaml_path)
shutil.copy2(yaml_path, yaml_backup)
print(f"✅ YAML backed up to: {yaml_backup}")
print("\nGenerated YAML:")
with open(yaml_path, 'r') as f:
    print(f.read())

## 8 — Verify Dataset Structure

In [ ]:
print("="*50 + "\n DATASET VERIFICATION\n" + "="*50)
ok = verify_dataset_structure(YOLO_DATASET_DIR)
print("\n🎯 Dataset ready!" if ok else "\n⚠️ Fix issues above.")

## 9 — Visualise Sample Annotations

In [ ]:
import cv2, matplotlib.pyplot as plt, random

train_labels_dir = os.path.join(YOLO_DATASET_DIR, 'labels', 'train')
train_images_dir = os.path.join(YOLO_DATASET_DIR, 'images', 'train')
label_files = [f for f in os.listdir(train_labels_dir) if f.endswith('.txt')]
sample_labels = random.sample(label_files, min(4, len(label_files)))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()
colors = [(255,80,80),(255,160,60),(60,180,255),(100,100,255),(255,220,60),
          (180,100,255),(100,255,100),(255,100,200),(0,255,200),(200,200,0)]

for idx, lf in enumerate(sample_labels):
    img_path = os.path.join(train_images_dir, lf.replace('.txt', '.jpg'))
    img = cv2.imread(img_path)
    if img is None:
        img = cv2.imread(img_path.replace('.jpg', '.png'))
    if img is None:
        axes[idx].set_title(f"Could not load"); continue
    h, w = img.shape[:2]
    with open(os.path.join(train_labels_dir, lf), 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            xc,yc,bw,bh = [float(v) for v in parts[1:5]]
            x1,y1 = int((xc-bw/2)*w), int((yc-bh/2)*h)
            x2,y2 = int((xc+bw/2)*w), int((yc+bh/2)*h)
            color = colors[cls_id % len(colors)]
            cv2.rectangle(img,(x1,y1),(x2,y2),color,2)
            cv2.putText(img,BDD_CLASSES[cls_id],(x1,y1-5),cv2.FONT_HERSHEY_SIMPLEX,0.5,color,1)
    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(lf.replace('.txt',''), fontsize=9)
    axes[idx].axis('off')
plt.suptitle('Sample BDD100K Annotations (YOLO format)', fontsize=13)
plt.tight_layout()
plt.show()

## 10 — Summary

In [ ]:
print("\n" + "="*60)
print(" BDD100K PREPARATION — COMPLETE")
print("="*60)
print(f" YOLO dataset dir:  {YOLO_DATASET_DIR}")
print(f" Dataset YAML:      {yaml_path}")
print(f" Train images:      {n_train}")
print(f" Val images:        {n_val}")
print(f" Classes:           {len(BDD_CLASSES)}")
print("="*60)
print("\n🎯 Proceed to notebook 03!")